# Homework: Linear Regression vs. Poisson GLM for Count Data
## Machine Learning Methods in Econometrics

In many real-world applications, the variable of interest is a **count** rather than a continuous quantity. Examples include the number of customer arrivals, insurance claims, equipment failures, and bicycle rentals. In such settings, standard linear regression may be hard to justify because the response is non-negative, discrete, and often highly skewed.

In this notebook, you will study **daily bicycle rental demand**. Predicting rental counts is important for bike-sharing operators and city planners: it helps them decide how many bicycles should be available, how to allocate bicycles across stations, when maintenance teams are needed, and how weather conditions affect demand.

The goal of this exercise is to compare two approaches for modeling count outcomes:

- a linear regression model under Gaussian assumptions,
- a Poisson generalized linear model (GLM) with a log link.

You should first implement the main steps **yourself**, using only basic numerical tools such as `numpy`. At the end, you will compare your implementation with the output of a standard library.

**Suggested grading: 100 points**
- Section 1. Load and inspect the data: **10 points**
- Section 2. Distribution analysis and KS statistic: **20 points**
- Section 3. Linear regression from scratch: **20 points**
- Section 4. Poisson GLM from scratch: **25 points**
- Section 5. Comparison of the two models: **10 points**
- Section 6. Verification using a library: **15 points**


Student names:

Student numbers:


## 1. Load and inspect the data [10 points]

The file `daily-bike-share.csv` is provided with the assignment. In this section, load the dataset, inspect the available columns, and choose a set of explanatory variables that you will use throughout the notebook.

A good starting set is:
- `temp`
- `hum`
- `windspeed`
- `workingday`
- `holiday`
- `weathersit`
- `season`

You may also experiment with other variables, but keep your final model interpretable.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

# Load the dataset from the local CSV file.
data = pd.read_csv("daily-bike-share.csv")

# TODO: Display the first few rows and inspect the columns.


In [ ]:

# TODO: display the list of columns and the shape of the dataset.
# Your code here


Throughout the notebook, let the response variable be `rentals`, the daily number of bicycle rentals. Since this is a count outcome, it is natural to ask whether a Gaussian model is appropriate or whether a count-data model may be more suitable.


In [ ]:

# TODO: define the response vector y and a design matrix X based on your chosen variables.
# Hint: keep a copy of the names of the selected features in a Python list.

feature_names = [
    # TODO
]

# X_raw should contain only the selected explanatory variables.
X_raw = None

# y should contain the rental counts.
y = None

# Display the first rows of X_raw if you want to inspect your chosen variables.


## 2. Distribution analysis and KS statistic [20 points]

Before fitting any model, study the distribution of the response variable.

A first question is whether the response looks approximately Gaussian. Since linear regression is commonly motivated by Gaussian errors and symmetric continuous outcomes, this is an important diagnostic step. Here you will:

1. inspect the distribution of `rentals`,
2. standardize the response,
3. compute the Kolmogorov--Smirnov statistic against the standard normal distribution.

Recall that if $z_{(1)} \leq \cdots \leq z_{(n)}$ are the ordered standardized observations, the KS statistic is

$$
D_n = \max\left\{
\max_i \left(\frac{i}{n} - \Phi(z_{(i)})\right),
\max_i \left(\Phi(z_{(i)}) - \frac{i-1}{n}\right)
\right\},
$$

where $\Phi$ is the cdf of the standard normal distribution.


In [ ]:

# TODO: plot a histogram of `rentals`.
# You may use around 30 bins.
# Your code here


Now standardize the response:

$$
z_i = \frac{y_i - \bar y}{s_y}.
$$

Then sort the standardized values and compute the KS statistic **manually**, without using a built-in KS test function.


In [ ]:

# Standardize the response
z = (y - y.mean()) / y.std(ddof=0)
z_sorted = np.sort(z)
n = len(z_sorted)

# TODO: compute the KS statistic D_n manually.
# Hint:
#   F_emp_upper at z_(i) is (i+1)/n in Python indexing,
#   F_emp_lower at z_(i) is i/n,
#   F_theory is norm.cdf(z_sorted[i]).

Dn = None

# Your code here

print("KS statistic:", Dn)


Briefly comment on the result. Does the empirical distribution of `rentals` look close to a Gaussian distribution? What does this suggest about the suitability of linear regression as a primary model?


Before fitting any model, split the sample into a training set and a test set.

Use:
- **80% of the observations for training**, and
- **20% for testing**.

To make your results reproducible, use a fixed random seed. You may generate a random permutation of the row indices and then assign the first 80% to the training set and the remaining 20% to the test set.

All model fitting in the rest of the notebook should be done **only on the training set**. The final **MAE** and **MSE** must be reported **on the test set**.


In [ ]:
# TODO: create an 80/20 train-test split.
# Suggested approach:
# 1. set a random seed,
# 2. randomly permute the row indices,
# 3. use the first 80% for training and the remaining 20% for testing.



# split_idx = ...
# train_idx = ...
# test_idx = ...

# Create train/test versions of the response for later use.
# y_train = ...
# y_test = ...

print("Number of training observations:", len(y_train))
print("Number of test observations:", len(y_test))


## 3. Linear regression from scratch [20 points]

As a benchmark, fit a linear regression model using the explanatory variables you selected.

After adding an intercept term, the ordinary least squares estimator is

$$
\hat\beta_{\mathrm{OLS}} = (X_{\mathrm{train}}^\top X_{\mathrm{train}})^{-1} X_{\mathrm{train}}^\top y_{\mathrm{train}}.
$$

In this section, compute the estimator directly using matrix algebra. Do not call a high-level regression-fitting library yet.

Important: fit the model **only on the training set**, then compute predictions on the **test set**.


In [ ]:
# TODO:
# Convert the raw feature table into a numeric matrix and add an intercept column.
# If you included categorical variables such as season or weathersit, convert them to dummies first and call it X_model.

# Your code here
X_model = None
X = None

# TODO: use train_idx and test_idx to create design matrices for the train and test sets.
# X_train = ...
# X_test = ...

print("Full design matrix shape:", X.shape)
print("Training design matrix shape:", X_train.shape)
print("Test design matrix shape:", X_test.shape)
print("Columns used in the design matrix:")
print(["const"] + X_model.columns.tolist())


In [ ]:
# TODO: compute the OLS estimator beta_ols using only the training set,
# and then compute predictions on both the training and test sets.
# Your code here

beta_ols = None
y_hat_ols_train = None
y_hat_ols_test = None

print("OLS coefficients:")
print(beta_ols)


In [ ]:
# TODO: compute two prediction metrics for OLS on the TEST set:
# - mean squared error
# - mean absolute error
#
# You may also compute training-set metrics if you wish, but the reported
# predictive comparison should be based on the test set.

mse_ols_test = None
mae_ols_test = None

# Your code here


After fitting the model, inspect the fitted values. Are any of the **test-set** OLS predictions negative? Why might this be problematic in a count-data application?


## 4. Poisson GLM from scratch [25 points]

Now fit a Poisson generalized linear model. For each observation $i$, assume

$$
Y_i \sim \mathrm{Poisson}(\mu_i),
\qquad
\mu_i = \exp(x_i^\top \beta).
$$

The log link guarantees that fitted means are positive. The log-likelihood (up to an additive constant) is

$$
\ell(\beta) = \sum_{i=1}^n \left( y_i x_i^\top \beta - \exp(x_i^\top \beta) \right).
$$

Fit this model **using the training set only**, and evaluate its predictive accuracy on the **test set**.


In [ ]:
# TODO: implement Newton iterations for the Poisson log-likelihood on the TRAINING set.
# Suggested initialization: beta = np.zeros(X_train.shape[1])
# Suggested stopping rule: stop when the Euclidean norm of the update is sufficiently small.

beta_pois = np.zeros(X_train.shape[1])

max_iter = 100
tol = 1e-8

for it in range(max_iter):
    # TODO: compute eta, mu, gradient, Hessian, and the Newton step using X_train and y_train.
    # Then update beta_pois.
    
    # Hint: Since mu = exp(eta), very large or very small eta values can cause
    # numerical overflow or underflow. Clipping eta to a range like [-20, 20]
    # keeps exp(eta) numerically stable.

    # eta = ...
    # mu = ...
    # gradient = ...
    # Hessian = ...
    # step = ...
    # beta_pois = ...

    # TODO: stopping rule
    pass

print("Poisson coefficients:")
print(beta_pois)


In [ ]:
# TODO: compute the fitted means for the Poisson model on both train and test sets.
# Then compute MSE and MAE on the TEST set for comparison with OLS.

y_hat_pois_train = None
y_hat_pois_test = None

mse_pois_test = None
mae_pois_test = None

# Your code here


Comment on the fitted values of the Poisson model on the **test set**. In what sense are they more natural for count data than the OLS predictions?


## 5. Compare the two models [10 points]

In this section, compare the fitted values and prediction quality of the two models. You may compare them numerically and visually.

A useful way to think about the difference is the following:

- OLS models the conditional mean as a linear function of the covariates.
- Poisson GLM models the conditional mean as $\exp(x_i^\top \beta)$.

This difference affects both the sign of the predictions and the way covariates enter the model.


In [ ]:
# TODO: print the TEST-SET metrics for both models in a clean format.
# Your code here


In [ ]:
# TODO: make two comparison plots using the TEST set.
# - true rentals versus OLS test predictions
# - true rentals versus Poisson test predictions

# Your code here


Based on your implementation, which model appears more appropriate for this application? Give a short explanation based on both the data, the model assumptions, and the **test-set** prediction results.


## 6. Verification using a library [15 points]

You have now implemented both models yourself. In this final section, compare your results with standard software.

Use a library implementation to fit:
- a linear regression model,
- a Poisson GLM.

Both library models should be fit on the **same training set** as before. Then compare:
- the estimated coefficients,
- the test-set predictions,
- and the test-set metrics you computed earlier.

Minor differences are acceptable and may arise from numerical optimization details.


In [ ]:
import statsmodels.api as sm

# TODO Fit OLS using a library on the TRAINING set.
ols_lib = None

# TODO Fit Poisson GLM using a library on the TRAINING set.
pois_lib = None

print(ols_lib.summary())
print(pois_lib.summary())


In [ ]:
# TODO: compare your coefficients and TEST-SET predictions with the library results.
# For example, you may compare using:
# - np.max(np.abs(beta_ols - ols_lib.params))
# - np.max(np.abs(beta_pois - pois_lib.params))
# - np.max(np.abs(y_hat_ols_test - ols_lib.predict(X_test)))
# - np.max(np.abs(y_hat_pois_test - pois_lib.predict(X_test)))

# Your code here


## Final discussion

Write a short concluding paragraph addressing the following points:

1. What did the KS statistic suggest about the distribution of the response?
2. What is the main conceptual difference between linear regression and the Poisson GLM in this application?
3. Which model appears more appropriate for predicting bicycle rental counts, and why?
4. How close were your manual implementations to the library results?
